<a href="https://colab.research.google.com/github/paul-444/learningStuff/blob/main/notebooks/unit8/unit8_part1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Unit 8: Proximal Policy Gradient (PPO) with PyTorch 🤖

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit9/thumbnail.png" alt="Unit 8"/>


In this notebook, you'll learn to **code your PPO agent from scratch with PyTorch using CleanRL implementation as model**.

To test its robustness, we're going to train it in:

- [LunarLander-v2 🚀](https://www.gymlibrary.dev/environments/box2d/lunar_lander/)


⬇️ Here is an example of what you will achieve. ⬇️

In [ ]:
%%html
<video controls autoplay><source src="https://huggingface.co/sb3/ppo-LunarLander-v2/resolve/main/replay.mp4" type="video/mp4"></video>

We're constantly trying to improve our tutorials, so **if you find some issues in this notebook**, please [open an issue on the GitHub Repo](https://github.com/huggingface/deep-rl-class/issues).

## Objectives of this notebook 🏆

At the end of the notebook, you will:

- Be able to **code your PPO agent from scratch using PyTorch**.
- Be able to **push your trained agent and the code to the Hub** with a nice video replay and an evaluation score 🔥.




## This notebook is from the Deep Reinforcement Learning Course
<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/deep-rl-course-illustration.jpg" alt="Deep RL Course illustration"/>

In this free course, you will:

- 📖 Study Deep Reinforcement Learning in **theory and practice**.
- 🧑‍💻 Learn to **use famous Deep RL libraries** such as Stable Baselines3, RL Baselines3 Zoo, CleanRL and Sample Factory 2.0.
- 🤖 Train **agents in unique environments**

Don’t forget to **<a href="http://eepurl.com/ic5ZUD">sign up to the course</a>** (we are collecting your email to be able to **send you the links when each Unit is published and give you information about the challenges and updates).**


The best way to keep in touch is to join our discord server to exchange with the community and with us 👉🏻 https://discord.gg/ydHrjt3WP5

## Prerequisites 🏗️
Before diving into the notebook, you need to:

🔲 📚 Study [PPO by reading Unit 8](https://huggingface.co/deep-rl-course/unit8/introduction) 🤗  

To validate this hands-on for the [certification process](https://huggingface.co/deep-rl-course/en/unit0/introduction#certification-process), you need to push one model, we don't ask for a minimal result but we **advise you to try different hyperparameters settings to get better results**.

If you don't find your model, **go to the bottom of the page and click on the refresh button**

For more information about the certification process, check this section 👉 https://huggingface.co/deep-rl-course/en/unit0/introduction#certification-process

## Set the GPU 💪
- To **accelerate the agent's training, we'll use a GPU**. To do that, go to `Runtime > Change Runtime type`

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/gpu-step1.jpg" alt="GPU Step 1">

- `Hardware Accelerator > GPU`

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/gpu-step2.jpg" alt="GPU Step 2">

## Create a virtual display 🔽

During the notebook, we'll need to generate a replay video. To do so, with colab, **we need to have a virtual screen to be able to render the environment** (and thus record the frames).

Hence the following cell will install the librairies and create and run a virtual screen 🖥

In [ ]:
!pip install setuptools==65.5.0

In [ ]:
%%capture
!apt install python-opengl
!apt install ffmpeg
!apt install xvfb
!apt install swig cmake build-essential
!pip install pyglet==1.5
!pip3 install pyvirtualdisplay

In [ ]:
# Virtual display
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

## Install dependencies 🔽
For this exercise, we use `gym==0.22`.

In [ ]:
!pip install gymnasium
!pip install box2d-py==2.3.5
!pip install imageio-ffmpeg
!pip install huggingface_hub

## Let's code PPO from scratch with Costa Huang tutorial
- For the core implementation of PPO we're going to use the excellent [Costa Huang](https://costa.sh/) tutorial.
- In addition to the tutorial, to go deeper you can read the 37 core implementation details: https://iclr-blog-track.github.io/2022/03/25/ppo-implementation-details/

👉 The video tutorial: https://youtu.be/MEt6rrxH8W4

In [ ]:
from IPython.display import HTML

HTML('<iframe width="560" height="315" src="https://www.youtube.com/embed/MEt6rrxH8W4" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture" allowfullscreen></iframe>')

- The best is to code first on the cell below, this way, if you kill the machine **you don't loose the implementation**.

In [ ]:
!pip uninstall -y pygame
!pip install pygame-ce

In [ ]:
!pip install swig
!pip install "gymnasium[box2d]"

In [ ]:
### Your code here:

In [ ]:
import random
import torch
import numpy as np
import gymnasium as gym

In [ ]:
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.backends.cudnn.deterministic=True
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
env=gym.make('CartPole-v1',render_mode='rgb_array')
env=gym.wrappers.RecordEpisodeStatistics(env)
env=gym.wrappers.RecordVideo(env,'videos',episode_trigger=lambda t:t%100==0)

In [ ]:
observation,info=env.reset()

In [ ]:
for _ in range(200):
  action=env.action_space.sample()
  observation,reward,terminated,truncated,info=env.step(action)
  if terminated or truncated:
    print(f'episodic return: {info['episode']['r']}')
    observation,info=env.reset()
env.close()

In [ ]:
def make_env(gym_id):
  def thunk():
    env=gym.make(gym_id,render_mode='rgb_array')
    env=gym.wrappers.RecordEpisodeStatistics(env)
    env=gym.wrappers.RecordVideo(env,'videos3',episode_trigger=lambda t: t%100==0)
    return env
  return thunk

In [ ]:
envs=gym.vector.SyncVectorEnv([make_env('CartPole-v1') for _ in range(2)])

In [ ]:
observation,info=envs.reset()

In [ ]:
for _ in range(200):
  action=envs.action_space.sample()
  observation,reward,terminated,truncated,info=envs.step(action)
  if '_episode' in info.keys():
    for env_idx,finished in enumerate(info['_episode']):
      if finished:
        print(f'env {env_idx} episode reward: {info['episode']['r'][env_idx]}')


In [ ]:
NUM_STEPS=512
NUM_ENVS=10
TOTAL_TIMESTEPS=int(1e6)
BATCH_SIZE=NUM_STEPS*NUM_ENVS
ANNEAL_LR=True
LEARNING_RATE=3e-4
GAE=True
GAMMA=0.99
GAE_LAMBDA=0.95
NUM_MINIBATCHES=4
MINIBATCH_SIZE=BATCH_SIZE//NUM_MINIBATCHES
UPDATE_EPOCHS=4
NORM_ADV=True
CLIP_COEF=0.2
CLIP_VLOSS=True
ENT_COEF=0.01
VF_COEF=0.5
MAX_GRAD_NORM=0.5
TARGET_KL=None

In [ ]:
envs=gym.vector.SyncVectorEnv([make_env('LunarLander-v3') for _ in range(NUM_ENVS)])

In [ ]:
import torch
from torch import nn
from torch.distributions.categorical import Categorical
import time

In [ ]:
def layer_init(layer,std=np.sqrt(2),bias_const=0.0):
  torch.nn.init.orthogonal_(layer.weight,std)
  torch.nn.init.constant_(layer.bias,bias_const)
  return layer

In [ ]:
class Agent(nn.Module):
  def __init__(self,envs):
    super(Agent,self).__init__()

    self.critic=nn.Sequential(
        layer_init(nn.Linear(np.array(envs.single_observation_space.shape).prod(),64)),
        nn.Tanh(),
        layer_init(nn.Linear(64,64)),
        nn.Tanh(),
        layer_init(nn.Linear(64,1),std=1.)
    )

    self.actor=nn.Sequential(
        layer_init(nn.Linear(np.array(envs.single_observation_space.shape).prod(),64)),
        nn.Tanh(),
        layer_init(nn.Linear(64,64)),
        nn.Tanh(),
        layer_init(nn.Linear(64,envs.single_action_space.n),std=1e-2)
    )

  def get_value(self,x):
    return self.critic(x)

  def get_action_and_value(self,x,action=None):
    logits=self.actor(x)
    probs=Categorical(logits=logits)
    if action is None:
      action=probs.sample()
    return action,probs.log_prob(action),probs.entropy(),self.critic(x)


In [ ]:
agent=Agent(envs).to(device)

In [ ]:
optimizer=torch.optim.Adam(agent.parameters(),lr=LEARNING_RATE,eps=1e-5)

In [ ]:
obs=torch.zeros((NUM_STEPS,NUM_ENVS)+envs.single_observation_space.shape).to(device)
actions=torch.zeros((NUM_STEPS,NUM_ENVS)+envs.single_action_space.shape).to(device)
logprobs=torch.zeros((NUM_STEPS,NUM_ENVS)).to(device)
rewards=torch.zeros((NUM_STEPS,NUM_ENVS)).to(device)
terminateds=torch.zeros((NUM_STEPS,NUM_ENVS)).to(device)
truncateds=torch.zeros((NUM_STEPS,NUM_ENVS)).to(device)
values=torch.zeros((NUM_STEPS,NUM_ENVS)).to(device)

In [ ]:
global_step=0
start_time=time.time()
next_obs=torch.Tensor(envs.reset()[0]).to(device)
next_terminated=torch.zeros(NUM_ENVS).to(device)
next_truncated=torch.zeros(NUM_ENVS).to(device)
num_updates=TOTAL_TIMESTEPS//BATCH_SIZE

In [ ]:
for update in range(1,num_updates+1):
  if ANNEAL_LR:
    frac=1.0-(update-1.0)/num_updates
    lrnow=frac*LEARNING_RATE
    optimizer.param_groups[0]['lr']=lrnow

  for step in range(0,NUM_STEPS):
    global_step+=1*NUM_ENVS
    obs[step]=next_obs
    terminateds[step]=next_terminated
    truncateds[step]=next_truncated

    with torch.no_grad():
      action,logprob,_,value=agent.get_action_and_value(next_obs)
      values[step]=value.flatten()
    actions[step]=action
    logprobs[step]=logprob

    next_obs,reward,terminated,truncated,info=envs.step(action.cpu().numpy())
    rewards[step]=torch.tensor(reward).to(device).view(-1)
    next_obs,next_terminated,next_truncated=torch.tensor(next_obs).to(device),torch.tensor(terminated,dtype=torch.bool).to(device),torch.tensor(truncated,dtype=torch.bool).to(device)


    if "episode" in info:
        # Print returns for all parallel environments that just finished
        for env_idx, done_flag in enumerate(
            next_terminated | next_truncated
        ):  # or info['_episode']
            if done_flag:
                print(
                    f"global step:{global_step} | episodic return:{info['episode']['r'][env_idx]:.2f}"
                )

  dones = terminateds.bool() | truncateds.bool()
  next_done = next_terminated | next_truncated
  with torch.no_grad():
    next_value=agent.get_value(next_obs).reshape(-1)
    if GAE:
      advantages=torch.zeros_like(rewards).to(device)
      lastgaelam=0
      for t in reversed(range(NUM_STEPS)):
        if t==NUM_STEPS-1:
          nextnonterminal=1.0-next_done.float()
          nextvalues=next_value
        else:
          nextnonterminal=1.0-dones[t+1].float()
          nextvalues=values[t+1]
        delta=rewards[t]+GAMMA*nextvalues*nextnonterminal-values[t]
        advantages[t]=lastgaelam=delta+GAMMA*GAE_LAMBDA*nextnonterminal*lastgaelam
      returns=advantages+values
    else:
      returns = torch.zeros_like(rewards).to(device)
      for t in reversed(range(NUM_STEPS)):
        if t == NUM_STEPS - 1:
          nextnonterminal = 1.0 - next_done
          next_return = next_value
        else:
          nextnonterminal = 1.0 - dones[t + 1]
          next_return = returns[t + 1]
        returns[t] = rewards[t] + GAMMA * nextnonterminal * next_return
      advantages = returns - values

  b_obs = obs.reshape((-1,) + envs.single_observation_space.shape)
  b_logprobs = logprobs.reshape(-1)
  b_actions = actions.reshape((-1,) + envs.single_action_space.shape)
  b_advantages = advantages.reshape(-1)
  b_returns = returns.reshape(-1)
  b_values = values.reshape(-1)

  b_inds=np.arange(BATCH_SIZE)
  clipfracs=[]
  for epoch in range(UPDATE_EPOCHS):
    np.random.shuffle(b_inds)
    for start in range(0,BATCH_SIZE,MINIBATCH_SIZE):
      end=start+MINIBATCH_SIZE
      mb_inds=b_inds[start:end]

      _, newlogprob, entropy, newvalue=agent.get_action_and_value(
          b_obs[mb_inds],b_actions.long()[mb_inds]
      )
      logratio=newlogprob-b_logprobs[mb_inds]
      ratio=logratio.exp()

      with torch.no_grad():
        old_approx_kl=(-logratio).mean()
        approx_kl=((ratio-1)-logratio).mean()
        clipfracs+=[((ratio-1.0).abs()>CLIP_COEF).float().mean()]

      mb_advantages=b_advantages[mb_inds]
      if NORM_ADV:
        mb_advantages=(mb_advantages-mb_advantages.mean())/(mb_advantages.std()+1e-8)

      pg_loss1=-mb_advantages*ratio
      pg_loss2=-mb_advantages*torch.clamp(ratio,1-CLIP_COEF,1+CLIP_COEF)
      pg_loss=torch.max(pg_loss1,pg_loss2).mean()

      newvalue=newvalue.view(-1)
      if CLIP_VLOSS:
        v_loss_unclipped=(newvalue-b_returns[mb_inds])**2
        v_clipped=b_values[mb_inds]+torch.clamp(
            newvalue-b_values[mb_inds],
            -CLIP_COEF,
            CLIP_COEF
        )
        v_loss_clipped=(v_clipped-b_returns[mb_inds])**2
        v_loss_max=torch.max(v_loss_unclipped,v_loss_clipped)
        v_loss=0.5*v_loss_max.mean()
      else:
        v_loss=0.5*((newvalue-b_returns[mb_inds])**2).mean()

      entropy_loss=entropy.mean()
      loss=pg_loss-ENT_COEF*entropy_loss+v_loss*VF_COEF

      optimizer.zero_grad()
      loss.backward()
      nn.utils.clip_grad_norm_(agent.parameters(),MAX_GRAD_NORM)
      optimizer.step()

    if TARGET_KL is not None:
      if approx_kl>TARGET_KL:
        break

  y_pred,y_true=b_values.cpu().numpy(),b_returns.cpu().numpy()
  var_y=np.var(y_true)
  explained_var=np.nan if var_y==0 else 1-np.var(y_true-y_pred)/var_y



## Add the Hugging Face Integration 🤗
- In order to push our model to the Hub, we need to define a function `package_to_hub`

- Add dependencies we need to push our model to the Hub

In [ ]:
from huggingface_hub import HfApi, upload_folder
from huggingface_hub.repocard import metadata_eval_result, metadata_save

from pathlib import Path
import datetime
import tempfile
import json
import shutil
import imageio

from wasabi import Printer
msg = Printer()

- Add new argument in `parse_args()` function to define the repo-id where we want to push the model.

In [ ]:
# Adding HuggingFace argument
parser.add_argument("--repo-id", type=str, default="ThomasSimonini/ppo-CartPole-v1", help="id of the model repository from the Hugging Face Hub {username/repo_name}")

- Next, we add the methods needed to push the model to the Hub

- These methods will:
  - `_evalutate_agent()`: evaluate the agent.
  - `_generate_model_card()`: generate the model card of your agent.
  - `_record_video()`: record a video of your agent.

In [ ]:
import datetime
import json
import shutil
import tempfile
from pathlib import Path

import imageio
import numpy as np
import torch
from huggingface_hub import HfApi, metadata_eval_result, metadata_save, upload_folder
from wasabi import msg

# NumPy 2.0+ compatibility patch for legacy Gym
if not hasattr(np, "bool8"):
    np.bool8 = np.bool_

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def _get_reset_state(env):
    res = env.reset()
    return res[0] if isinstance(res, tuple) else res


def _step_env(env, action):
    res = env.step(action)
    if len(res) == 5:
        next_state, reward, terminated, truncated, _ = res
        done = terminated or truncated
    else:
        next_state, reward, done, _ = res
    return next_state, reward, done


def package_to_hub(
    repo_id,
    model,
    hyperparameters,
    eval_env,
    video_fps=30,
    commit_message="Push agent to the Hub",
    token=None,
    logs=None,
):
    msg.info(
        "This function will save, evaluate, generate a video of your agent, "
        "create a model card and push everything to the hub.\n"
        "This might take up to 1min."
    )

    env_id = (
        hyperparameters.get("env_id", "LunarLander-v2")
        if isinstance(hyperparameters, dict)
        else getattr(hyperparameters, "env_id", "LunarLander-v2")
    )

    repo_url = HfApi().create_repo(
        repo_id=repo_id,
        token=token,
        private=False,
        exist_ok=True,
    )

    with tempfile.TemporaryDirectory() as tmpdirname:
        tmpdirname = Path(tmpdirname)

        torch.save(model.state_dict(), tmpdirname / "model.pt")

        mean_reward, std_reward = _evaluate_agent(eval_env, 10, model)

        eval_datetime = datetime.datetime.now().isoformat()
        evaluate_data = {
            "env_id": env_id,
            "mean_reward": float(mean_reward),
            "std_reward": float(std_reward),
            "n_evaluation_episodes": 10,
            "eval_datetime": eval_datetime,
        }

        with open(tmpdirname / "results.json", "w") as outfile:
            json.dump(evaluate_data, outfile, indent=2)

        video_path = tmpdirname / "replay.mp4"
        record_video(eval_env, model, video_path, video_fps)

        generated_model_card, metadata = _generate_model_card(
            "PPO", env_id, mean_reward, std_reward, hyperparameters
        )
        _save_model_card(tmpdirname, generated_model_card, metadata)

        if logs:
            _add_logdir(tmpdirname, Path(logs))

        msg.info(f"Pushing repo {repo_id} to the Hugging Face Hub")

        repo_url = upload_folder(
            repo_id=repo_id,
            folder_path=tmpdirname,
            path_in_repo="",
            commit_message=commit_message,
            token=token,
        )

        msg.info(f"Your model is pushed to the Hub. View it here: {repo_url}")

    return repo_url


def _evaluate_agent(env, n_eval_episodes, policy):
    episode_rewards = []
    for _ in range(n_eval_episodes):
        state = _get_reset_state(env)
        done = False
        total_rewards_ep = 0.0

        while not done:
            state_tensor = torch.tensor(np.array(state), dtype=torch.float32).unsqueeze(0).to(device)

            with torch.no_grad():
                action, _, _, _ = policy.get_action_and_value(state_tensor)

            action_np = action.cpu().numpy()
            if action_np.ndim > 0:
                action_np = action_np[0]

            state, reward, done = _step_env(env, action_np)
            total_rewards_ep += float(reward)

        episode_rewards.append(total_rewards_ep)

    return float(np.mean(episode_rewards)), float(np.std(episode_rewards))


def record_video(env, policy, out_directory, fps=30):
    images = []
    done = False
    state = _get_reset_state(env)

    try:
        img = env.render(mode="rgb_array")
    except TypeError:
        img = env.render()

    if img is not None:
        images.append(img)

    while not done:
        state_tensor = torch.tensor(np.array(state), dtype=torch.float32).unsqueeze(0).to(device)

        with torch.no_grad():
            action, _, _, _ = policy.get_action_and_value(state_tensor)

        action_np = action.cpu().numpy()
        if action_np.ndim > 0:
            action_np = action_np[0]

        state, reward, done = _step_env(env, action_np)

        try:
            img = env.render(mode="rgb_array")
        except TypeError:
            img = env.render()

        if img is not None:
            images.append(img)

    if images:
        imageio.mimsave(out_directory, [np.array(i) for i in images], fps=fps)


def _generate_model_card(model_name, env_id, mean_reward, std_reward, hyperparameters):
    metadata = generate_metadata(model_name, env_id, mean_reward, std_reward)

    if isinstance(hyperparameters, dict):
        converted_dict = hyperparameters
    else:
        converted_dict = vars(hyperparameters)

    converted_str = "\n".join([f"{k}: {v}" for k, v in converted_dict.items()])

    ticks = "```"
    model_card = (
        f"# {model_name} Agent Playing {env_id}\n\n"
        f"This is a trained model of a {model_name} agent playing {env_id}.\n\n"
        f"## Hyperparameters\n"
        f"{ticks}yaml\n{converted_str}\n{ticks}\n"
    )
    return model_card, metadata


def generate_metadata(model_name, env_id, mean_reward, std_reward):
    metadata = {
        "tags": [
            env_id,
            "ppo",
            "deep-reinforcement-learning",
            "reinforcement-learning",
            "custom-implementation",
            "deep-rl-course",
        ]
    }

    eval_data = metadata_eval_result(
        model_pretty_name=model_name,
        task_pretty_name="reinforcement-learning",
        task_id="reinforcement-learning",
        metrics_pretty_name="mean_reward",
        metrics_id="mean_reward",
        metrics_value=f"{mean_reward:.2f} +/- {std_reward:.2f}",
        dataset_pretty_name=env_id,
        dataset_id=env_id,
    )

    return {**metadata, **eval_data}


def _save_model_card(local_path, generated_model_card, metadata):
    readme_path = local_path / "README.md"
    readme_path.write_text(generated_model_card, encoding="utf-8")
    metadata_save(readme_path, metadata)


def _add_logdir(local_path: Path, logdir: Path):
    if logdir.exists() and logdir.is_dir():
        repo_logdir = local_path / "logs"
        if repo_logdir.exists():
            shutil.rmtree(repo_logdir)
        shutil.copytree(logdir, repo_logdir)

- Finally, we call this function at the end of the PPO training

In [ ]:
config = {
    "env_id": "LunarLander-v2",
    "total_timesteps": 1000000,
    "learning_rate": 3e-4,
    "num_envs": 10,
    "num_steps": 512,
    "anneal_lr": True,
    "gae": True,
    "gamma": 0.99,
    "gae_lambda": 0.95,
    "num_minibatches": 4,
    "update_epochs": 4,
    "norm_adv": True,
    "clip_coef": 0.2,
    "clip_vloss": True,
    "ent_coef": 0.01,
    "vf_coef": 0.5,
    "max_grad_norm": 0.5,
    "repo_id": "ThomasSimonini/ppo-LunarLander-v2",
}

In [ ]:
from huggingface_hub import login

# This will prompt you to enter your Hugging Face token
login()

In [ ]:
# 1. Temporarily set env_id to LunarLander-v2 in your config for the grader
config["env_id"] = "LunarLander-v2"

# 2. Call package_to_hub standardly (WITHOUT hub_env_id)
package_to_hub(
    repo_id="paul-444/ppo-LunarLander-v2",  # Recommended: avoid parentheses in repo names
    model=agent,
    hyperparameters=config,
    eval_env=gym.make("LunarLander-v2"),     # Evaluates on v3 locally
    logs=f"runs/first",
)

- Here's what look the ppo.py final file

To be able to share your model with the community there are three more steps to follow:

1️⃣ (If it's not already done) create an account to HF ➡ https://huggingface.co/join

2️⃣ Sign in and then, you need to store your authentication token from the Hugging Face website.
- Create a new token (https://huggingface.co/settings/tokens) **with write role**

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/create-token.jpg" alt="Create HF Token">

- Copy the token
- Run the cell below and paste the token

In [ ]:
from huggingface_hub import notebook_login
notebook_login()
!git config --global credential.helper store

If you don't want to use a Google Colab or a Jupyter Notebook, you need to use this command instead: `huggingface-cli login`

## Let's start the training 🔥
- ⚠️ ⚠️ ⚠️  Don't use **the same repo id with the one you used for the Unit 1**
- Now that you've coded from scratch PPO and added the Hugging Face Integration, we're ready to start the training 🔥

- First, you need to copy all your code to a file you create called `ppo.py`

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit9/step1.png" alt="PPO"/>

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit9/step2.png" alt="PPO"/>

- Now we just need to run this python script using `python <name-of-python-script>.py` with the additional parameters we defined with `argparse`

- You should modify more hyperparameters otherwise the training will not be super stable.

In [ ]:
!python ppo.py --env-id="LunarLander-v2" --repo-id="YOUR_REPO_ID" --total-timesteps=50000

## Some additional challenges 🏆
The best way to learn **is to try things by your own**! Why not trying  another environment?


See you on Unit 8, part 2 where we going to train agents to play Doom 🔥
## Keep learning, stay awesome 🤗